# 🏪 Demand Forecasting – Micro-Fulfillment Hubs
## Main Workflow Notebook

**Task:** Predict daily `OrderVolume` per hub for the test period.
**Metric:** RMSPE (Root Mean Squared Percentage Error)
**Models:** LightGBM (primary) + XGBoost (comparison)

### Pipeline
1. Load & Merge Data
2. EDA
3. Feature Engineering
4. Time-based Train/Val Split
5. Model Training
6. Evaluation
7. Predictions & Submission
8. Feature Importance Analysis

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import sys
import os
from pathlib import Path

# Make src importable from notebooks/
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from utils    import load_data
from eda      import perform_eda
from features import create_features, get_feature_cols
from model    import (train_model, evaluate_model, predict,
                      save_model, get_feature_importance, rmspe)

# Paths
DATA_PATH      = PROJECT_ROOT / 'data'
OUTPUT_PATH    = PROJECT_ROOT / 'outputs'
PLOT_PATH      = OUTPUT_PATH  / 'plots'
ARTIFACT_PATH  = OUTPUT_PATH  / 'model_artifacts'
ARTIFACT_PATH.mkdir(parents=True, exist_ok=True)
PLOT_PATH.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print('✅ Setup complete')

---
## 1. Load & Merge Data

In [ ]:
# ============================================================
# 1. LOAD
# ============================================================
orders_train, orders_test, hub_metadata = load_data(DATA_PATH)

# Quick preview
print('\n--- orders_train head ---')
display(orders_train.head(3))
print('\n--- orders_test head ---')
display(orders_test.head(3))
print('\n--- hub_metadata head ---')
display(hub_metadata.head(3))

In [ ]:
# ============================================================
# 1b. MERGE with hub metadata
# ============================================================
train = orders_train.merge(hub_metadata, on='HubID', how='left')
test  = orders_test.merge( hub_metadata, on='HubID', how='left')

print(f'Merged train : {train.shape}')
print(f'Merged test  : {test.shape}')
display(train.dtypes)

---
## 2. Exploratory Data Analysis

In [ ]:
# ============================================================
# 2. EDA
# ============================================================
eda_results = perform_eda(
    train    = train,
    test     = test,
    target   = 'OrderVolume',
    hub_id   = 'HubID',
    date_col = 'Date',
    plot_path= PLOT_PATH,
)

print('\n--- Key Findings ---')
for i, f in enumerate(eda_results.get('key_findings', []), 1):
    print(f'  [{i}]', f)

---
## 3. Feature Engineering

In [ ]:
# ============================================================
# 3a. FEATURE ENGINEERING – Train
# ============================================================
train_feat = create_features(
    df         = train,
    date_col   = 'Date',
    hub_id     = 'HubID',
    target_col = 'OrderVolume',
    is_train   = True,
)

print(f'Train featured shape: {train_feat.shape}')
display(train_feat.head(2))

In [ ]:
# ============================================================
# 3b. FEATURE ENGINEERING – Test
# ============================================================
#
# For test we CANNOT compute lag features from the test set itself.
# Strategy: concatenate train+test, build lags on the full series,
# then separate them back. This propagates train lags into test rows.
# ============================================================

# Mark sets
train_tmp      = train.copy()
train_tmp['_set'] = 'train'
test_tmp       = test.copy()
test_tmp['_set'] = 'test'

# Add dummy OrderVolume to test so create_features doesn't fail
test_tmp['OrderVolume'] = np.nan
test_tmp['AppSessions'] = test_tmp.get('AppSessions', pd.Series(0, index=test_tmp.index)).fillna(0)

combined = pd.concat([train_tmp, test_tmp], ignore_index=True)
combined = combined.sort_values(['HubID', 'Date']).reset_index(drop=True)

combined_feat = create_features(
    df         = combined,
    date_col   = 'Date',
    hub_id     = 'HubID',
    target_col = 'OrderVolume',
    is_train   = True,   # use target col for lags; test rows will be NaN in target
)

train_feat = combined_feat[combined_feat['_set'] == 'train'].copy()
test_feat  = combined_feat[combined_feat['_set'] == 'test'].copy()

# Drop helper col
train_feat.drop(columns=['_set'], inplace=True)
test_feat.drop(columns=['_set'], inplace=True)

print(f'Train featured: {train_feat.shape}')
print(f'Test  featured: {test_feat.shape}')

---
## 4. Time-Based Train / Validation Split

In [ ]:
# ============================================================
# 4. TIME-BASED SPLIT  (80 / 20 expanding window)
# ============================================================
train_feat = train_feat.sort_values(['HubID', 'Date']).reset_index(drop=True)

unique_dates = np.sort(train_feat['Date'].unique())
split_idx    = int(len(unique_dates) * 0.80)
train_dates  = unique_dates[:split_idx]
val_dates    = unique_dates[split_idx:]

df_tr  = train_feat[train_feat['Date'].isin(train_dates)].copy()
df_val = train_feat[train_feat['Date'].isin(val_dates)].copy()

print(f'Train period : {pd.Timestamp(train_dates[0]).date()}  →  {pd.Timestamp(train_dates[-1]).date()}  ({len(df_tr):,} rows)')
print(f'Val   period : {pd.Timestamp(val_dates[0]).date()}  →  {pd.Timestamp(val_dates[-1]).date()}  ({len(df_val):,} rows)')

# Filter closed hubs (IsOpen=0) from training – they will always be 0
df_tr_open  = df_tr[df_tr['IsOpen'] == 1].copy()
df_val_open = df_val.copy()  # keep all for realistic eval

print(f'\nTrain (open only) : {len(df_tr_open):,} rows')

In [ ]:
# Feature columns
FEATURE_COLS = get_feature_cols(train_feat)
TARGET_COL   = 'OrderVolume'

print(f'Total features: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

In [ ]:
# Drop rows with NaN in lag features (first 28 days per hub)
lag_cols = [c for c in FEATURE_COLS if c.startswith('lag_')]
df_tr_clean = df_tr_open.dropna(subset=lag_cols).copy()

X_train = df_tr_clean[FEATURE_COLS]
y_train = df_tr_clean[TARGET_COL]

X_val   = df_val_open[FEATURE_COLS]
y_val   = df_val_open[TARGET_COL]

print(f'X_train : {X_train.shape}  |  X_val : {X_val.shape}')

---
## 5. Model Training

In [ ]:
# ============================================================
# 5a. LightGBM (Primary)
# ============================================================
print('=== Training LightGBM ===')
model_lgb = train_model(
    X_train    = X_train,
    y_train    = y_train,
    X_val      = X_val,
    y_val      = y_val,
    model_type = 'lightgbm',
    params     = {
        'n_estimators'    : 2000,
        'learning_rate'   : 0.03,
        'num_leaves'      : 63,
        'max_depth'       : -1,
        'subsample'       : 0.8,
        'colsample_bytree': 0.8,
        'min_child_samples': 20,
        'reg_alpha'       : 0.1,
        'reg_lambda'      : 0.1,
    },
    early_stopping_rounds = 50,
)

In [ ]:
# ============================================================
# 5b. XGBoost (Comparison)
# ============================================================
print('=== Training XGBoost ===')
model_xgb = train_model(
    X_train    = X_train,
    y_train    = y_train,
    X_val      = X_val,
    y_val      = y_val,
    model_type = 'xgboost',
    params     = {
        'n_estimators'    : 2000,
        'learning_rate'   : 0.03,
        'max_depth'       : 6,
        'subsample'       : 0.8,
        'colsample_bytree': 0.8,
    },
    early_stopping_rounds = 50,
)

---
## 6. Evaluation

In [ ]:
# ============================================================
# 6. EVALUATE on Validation Set
# ============================================================
print('=== LightGBM Validation ===')
val_preds_lgb = predict(model_lgb, X_val)
metrics_lgb   = evaluate_model(y_val, val_preds_lgb)

print('\n=== XGBoost Validation ===')
val_preds_xgb = predict(model_xgb, X_val)
metrics_xgb   = evaluate_model(y_val, val_preds_xgb)

print('\n=== Ensemble (avg) Validation ===')
val_preds_ens = 0.5 * val_preds_lgb + 0.5 * val_preds_xgb
metrics_ens   = evaluate_model(y_val, val_preds_ens)

In [ ]:
# -- Error Analysis Plots --
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, preds, name in zip(axes,
    [val_preds_lgb, val_preds_xgb, val_preds_ens],
    ['LightGBM', 'XGBoost', 'Ensemble']):

    ax.scatter(y_val, preds, alpha=0.05, s=3)
    max_v = max(y_val.max(), preds.max())
    ax.plot([0, max_v], [0, max_v], 'r--', lw=1, label='Perfect')
    ax.set_title(f'{name}\nRMSPE={rmspe(y_val, preds):.4f}')
    ax.set_xlabel('Actual OrderVolume')
    ax.set_ylabel('Predicted OrderVolume')
    ax.legend(fontsize=8)

plt.suptitle('Actual vs Predicted – Validation Set', fontsize=13)
plt.tight_layout()
plt.savefig(PLOT_PATH / '12_actual_vs_predicted.png', dpi=120)
plt.show()
print('[plot] saved → 12_actual_vs_predicted.png')

In [ ]:
# -- Residual distribution --
residuals = y_val.values - val_preds_lgb

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(residuals, bins=60, color='steelblue', edgecolor='none')
axes[0].axvline(0, color='red', lw=1.5, ls='--')
axes[0].set_title('Residuals Distribution (LightGBM)')
axes[0].set_xlabel('Residual (Actual - Predicted)')

axes[1].scatter(val_preds_lgb, residuals, alpha=0.05, s=3, color='steelblue')
axes[1].axhline(0, color='red', lw=1.5, ls='--')
axes[1].set_title('Residuals vs Fitted (LightGBM)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.savefig(PLOT_PATH / '13_residuals.png', dpi=120)
plt.show()

---
## 7. Feature Importance

In [ ]:
# ============================================================
# 7. FEATURE IMPORTANCE
# ============================================================
fi_lgb = get_feature_importance(model_lgb, FEATURE_COLS, model_type='lightgbm')
fi_xgb = get_feature_importance(model_xgb, FEATURE_COLS, model_type='xgboost')

TOP_N = 25

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, fi, name in zip(axes, [fi_lgb, fi_xgb], ['LightGBM', 'XGBoost']):
    top = fi.head(TOP_N)
    ax.barh(top['feature'][::-1], top['importance'][::-1], color='steelblue')
    ax.set_title(f'Top {TOP_N} Features – {name}')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig(PLOT_PATH / '14_feature_importance.png', dpi=120)
plt.show()
print('[plot] saved → 14_feature_importance.png')

print('\n--- LightGBM Top 15 Features ---')
display(fi_lgb.head(15))

---
## 8. Predictions & Submission

In [ ]:
# ============================================================
# 8. GENERATE PREDICTIONS
# ============================================================

# Fill any NaN in test features with 0 (lag cols won't exist for test rows
# that don't have 28 days of history available)
X_test = test_feat[FEATURE_COLS].fillna(0)

# Choose best model (lowest RMSPE)
best_rmspe  = min(metrics_lgb['rmspe'], metrics_xgb['rmspe'], metrics_ens['rmspe'])
if best_rmspe == metrics_ens['rmspe']:
    print('Using ENSEMBLE predictions')
    test_preds_lgb = predict(model_lgb, X_test)
    test_preds_xgb = predict(model_xgb, X_test)
    test_preds = 0.5 * test_preds_lgb + 0.5 * test_preds_xgb
elif best_rmspe == metrics_lgb['rmspe']:
    print('Using LightGBM predictions')
    test_preds = predict(model_lgb, X_test)
else:
    print('Using XGBoost predictions')
    test_preds = predict(model_xgb, X_test)

# For closed hubs: force prediction to 0
if 'IsOpen' in test_feat.columns:
    test_preds[test_feat['IsOpen'].values == 0] = 0

print(f'Test predictions – min: {test_preds.min():.2f}, max: {test_preds.max():.2f}, mean: {test_preds.mean():.2f}')

In [ ]:
# ============================================================
# 8b. CREATE SUBMISSION FILE  (format: Id, OrderVolume)
# ============================================================
submission = pd.DataFrame({
    'Id'         : test_feat['Id'].values,
    'OrderVolume': np.round(test_preds).astype(int),
})

submission_path = OUTPUT_PATH / 'predictions.csv'
submission.to_csv(submission_path, index=False)
print(f'✅ Submission saved → {submission_path}')
display(submission.head(10))

---
## 9. Save Artifacts

In [ ]:
# ============================================================
# 9. SAVE MODELS & ARTIFACTS
# ============================================================
save_model(model_lgb, ARTIFACT_PATH / 'lightgbm_model.pkl', FEATURE_COLS)
save_model(model_xgb, ARTIFACT_PATH / 'xgboost_model.pkl',  FEATURE_COLS)

# Save metrics summary
metrics_df = pd.DataFrame([
    {'model': 'LightGBM', **metrics_lgb},
    {'model': 'XGBoost',  **metrics_xgb},
    {'model': 'Ensemble', **metrics_ens},
])
metrics_df.to_csv(OUTPUT_PATH / 'validation_metrics.csv', index=False)
print('✅ Metrics saved → validation_metrics.csv')
display(metrics_df)

# Save feature importances
fi_lgb.to_csv(OUTPUT_PATH / 'feature_importance_lgb.csv', index=False)
fi_xgb.to_csv(OUTPUT_PATH / 'feature_importance_xgb.csv', index=False)
print('✅ Feature importances saved')

print('\n🚀 Pipeline complete!')